# PlantCare AI - retrain with field photos
Turn on **Runtime > Change runtime type > T4 GPU**, then run the cells in order.

In [ ]:
import torch; print("GPU:", torch.cuda.is_available())
from google.colab import files
up = files.upload()   # choose plant_care_ai.zip, train_field.py and export_onnx.py together
!unzip -q -o plant_care_ai.zip
!mv train_field.py export_onnx.py plant_care_ai/
%cd /content/plant_care_ai
!pip install -q -U kaggle onnx onnxscript
!mkdir -p models web

## 1. PlantVillage (needs a Kaggle token; paste it when asked)

In [ ]:
import os, getpass, glob
os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Paste Kaggle token: ")
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p data --unzip
DATA_DIR = glob.glob("data/**/color", recursive=True)[0]; print(DATA_DIR)

## 2. PlantDoc field photos (CC BY 4.0)

In [ ]:
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git
!ls PlantDoc-Dataset/train

## 3. Merge and map class names (check the 'unmapped' line in the output)

In [ ]:
MAP = {  # PlantDoc folder name (lower case) -> PlantVillage class. Fix any names printed as "unmapped".
 "apple scab leaf":"Apple___Apple_scab","apple rust leaf":"Apple___Cedar_apple_rust","apple leaf":"Apple___healthy",
 "bell_pepper leaf spot":"Pepper,_bell___Bacterial_spot","bell_pepper leaf":"Pepper,_bell___healthy","blueberry leaf":"Blueberry___healthy",
 "cherry leaf":"Cherry_(including_sour)___healthy","corn gray leaf spot":"Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
 "corn leaf blight":"Corn_(maize)___Northern_Leaf_Blight","corn rust leaf":"Corn_(maize)___Common_rust_",
 "grape leaf":"Grape___healthy","grape leaf black rot":"Grape___Black_rot","peach leaf":"Peach___healthy",
 "potato leaf early blight":"Potato___Early_blight","potato leaf late blight":"Potato___Late_blight","raspberry leaf":"Raspberry___healthy",
 "soyabean leaf":"Soybean___healthy","squash powdery mildew leaf":"Squash___Powdery_mildew","strawberry leaf":"Strawberry___healthy",
 "tomato early blight leaf":"Tomato___Early_blight","tomato leaf":"Tomato___healthy","tomato leaf bacterial spot":"Tomato___Bacterial_spot",
 "tomato leaf late blight":"Tomato___Late_blight","tomato leaf mosaic virus":"Tomato___Tomato_mosaic_virus",
 "tomato leaf yellow virus":"Tomato___Tomato_Yellow_Leaf_Curl_Virus","tomato mold leaf":"Tomato___Leaf_Mold",
 "tomato septoria leaf spot":"Tomato___Septoria_leaf_spot","tomato two spotted spider mites leaf":"Tomato___Spider_mites Two-spotted_spider_mite"}
import os, random, shutil
from pathlib import Path
random.seed(0)
PV, PD, OUT, FIELD = Path(DATA_DIR), Path("PlantDoc-Dataset"), Path("merged"), Path("field")
shutil.rmtree(OUT, ignore_errors=True); shutil.rmtree(FIELD, ignore_errors=True)
for cls in sorted(os.listdir(PV)):                      # 1) up to 400 PlantVillage lab photos per class
    files = [f for f in (PV/cls).iterdir() if f.is_file()]; random.shuffle(files)
    (OUT/cls).mkdir(parents=True, exist_ok=True)
    for f in files[:400]: shutil.copy(f, OUT/cls/f.name)
unmapped = set()
for split, dest, reps in (("train", OUT, 3), ("test", FIELD, 1)):   # 2) PlantDoc: train x3 into merged, test -> field
    for d in (PD/split).iterdir():
        if not d.is_dir(): continue
        if d.name.lower() not in MAP: unmapped.add(d.name); continue
        tgt = dest/MAP[d.name.lower()]; tgt.mkdir(parents=True, exist_ok=True)
        for f in d.iterdir():
            for r in range(reps): shutil.copy(f, tgt/f"pd{r}_{f.name}")
print("unmapped PlantDoc folders (ignored):", sorted(unmapped))
print("field classes:", len(os.listdir(FIELD)), " field photos:", sum(len(os.listdir(FIELD/c)) for c in os.listdir(FIELD)))

## 4. Train with strong field-style augmentation (about 15-30 min)

In [ ]:
!python train_field.py --train-dir merged --field-dir field --epochs-head 2 --epochs-ft 8 --num-workers 2

## 5. Export the browser model and download it

In [ ]:
!python export_onnx.py
from google.colab import files
files.download("plant_classifier.onnx"); files.download("classes.json")